# Module 6 실습: Sampling · Elicitation · Roots with FastMCP

이 노트북은 **Module 6: 고급 상호작용 패턴 - Sampling 및 Elicitation, Roots** 실습용입니다.

학습 목표:
- **Sampling**: 서버가 클라이언트(Host)의 LLM 기능을 역으로 요청하는 흐름 이해
- **Elicitation**: 서버가 사용자에게 부족한 정보를 구조적으로 요청하는 흐름 이해
- **Roots**: 클라이언트가 서버에 허용된 파일 시스템 경계를 알려 주는 방식 이해

이번 실습은 Colab에서 실행되도록 구성되어 있습니다.  
Colab은 Claude Desktop 같은 완전한 MCP Host가 아니므로, **FastMCP Client의 callback handler가 Host 역할을 시뮬레이션**합니다.


## 실행 순서

아래 순서대로 실행하세요.

1. **FastMCP 설치**
2. **기본 import**
3. **샘플 workspace(루트 디렉터리) 생성**
4. **MCP Server 정의**
5. **Sampling / Elicitation / Roots handler 정의**
6. **Client 생성**
7. **Roots 확인**
8. **루트 내부 파일 읽기 + Sampling 요약**
9. **Elicitation으로 추가 입력 수집 + Sampling으로 계획 생성**
10. **루트 경계 위반 시도 확인**
11. **실습 정리**

권장:
- 메뉴에서 **런타임 → 모두 실행**
- 또는 위에서 아래로 한 셀씩 실행


## 핵심 개념 요약

### Sampling
Sampling은 서버가 클라이언트에게 **LLM 생성 작업을 요청**하는 기능입니다.  
서버는 자체 API 키 없이도, 클라이언트가 가진 모델 접근 권한을 활용해 요약·추론·생성을 요청할 수 있습니다.

### Elicitation
Elicitation은 서버가 **사용자에게 추가 정보를 요청**하는 기능입니다.  
부족한 파라미터가 있을 때, 서버가 실행을 멈추고 구조화된 질문을 던진 뒤 응답을 받아 계속 진행할 수 있습니다.

### Roots
Roots는 클라이언트가 서버에 **접근 가능한 파일 시스템 경계**를 알려 주는 기능입니다.  
서버는 이 정보를 바탕으로 허용된 디렉터리 안에서만 작업하도록 설계할 수 있습니다.


In [ ]:
# 1) 설치
!pip -q install -U fastmcp


In [ ]:
# 2) 기본 import
from dataclasses import dataclass, fields, is_dataclass
from pathlib import Path
from urllib.parse import urlparse, unquote
import json

from fastmcp import FastMCP, Client, Context

print("import 완료")


## 1) 샘플 workspace 생성

Roots 실습을 위해, 클라이언트가 서버에 노출할 **허용된 루트 디렉터리**를 하나 만듭니다.


In [ ]:
WORKSPACE_DIR = Path("module6_workspace").resolve()
WORKSPACE_DIR.mkdir(exist_ok=True)

note_path = WORKSPACE_DIR / "project_note.txt"
note_path.write_text(
    "MCP Module 6\n"
    "Sampling은 서버가 Host의 LLM에 생성 작업을 요청하는 기능입니다.\n"
    "Elicitation은 서버가 사용자에게 부족한 정보를 구조적으로 요청하는 기능입니다.\n"
    "Roots는 서버가 접근할 수 있는 파일 시스템 경계를 정의합니다.\n",
    encoding="utf-8"
)

print("Workspace:", WORKSPACE_DIR)
print("Created file:", note_path)
print(note_path.read_text(encoding="utf-8"))


## 2) MCP Server 정의

이번 실습 서버는 다음 세 가지 tool을 제공합니다.

1. `show_roots`  
   서버가 클라이언트가 노출한 roots를 확인합니다.

2. `summarize_note_from_root`  
   roots 내부 파일만 읽고, 그 내용을 Sampling으로 요약합니다.

3. `interactive_project_brief`  
   Elicitation으로 추가 입력을 수집한 뒤, Sampling으로 계획 초안을 생성합니다.


In [ ]:
mcp = FastMCP("Module6-Advanced-Patterns-Demo")

@dataclass
class PlanInput:
    deadline: str
    priority: str
    assumptions: str

def _root_to_path(root_obj) -> Path:
    """
    FastMCP Root 객체 또는 문자열을 Path로 변환합니다.
    file:// URI와 일반 경로 문자열 둘 다 처리합니다.
    """
    uri = getattr(root_obj, "uri", None)
    if uri is None:
        uri = str(root_obj)

    if uri.startswith("file://"):
        parsed = urlparse(uri)
        return Path(unquote(parsed.path)).resolve()
    return Path(uri).resolve()

@mcp.tool
async def show_roots(ctx: Context) -> dict:
    """클라이언트가 노출한 roots 목록을 반환합니다."""
    roots = ctx.list_roots()
    root_list = [str(_root_to_path(r)) for r in roots]
    return {
        "root_count": len(root_list),
        "roots": root_list
    }

@mcp.tool
async def summarize_note_from_root(filename: str, ctx: Context) -> dict:
    """허용된 root 안의 파일만 읽고, Sampling으로 요약합니다."""
    roots = ctx.list_roots()
    if not roots:
        return {"ok": False, "error": "No roots provided by client."}

    root_path = _root_to_path(roots[0])
    candidate = (root_path / filename).resolve()

    try:
        candidate.relative_to(root_path)
    except ValueError:
        return {
            "ok": False,
            "error": "Requested path is outside the allowed root boundary.",
            "requested": str(candidate),
            "allowed_root": str(root_path),
        }

    if not candidate.exists():
        return {
            "ok": False,
            "error": "File not found inside allowed root.",
            "requested": str(candidate),
        }

    content = candidate.read_text(encoding="utf-8")

    sampling_result = await ctx.sample(
        messages=(
            "Please summarize the following note in clear Korean.\n\n"
            f"{content}"
        ),
        system_prompt="You are a concise note summarizer."
    )

    return {
        "ok": True,
        "root": str(root_path),
        "file": str(candidate),
        "summary": sampling_result.text or sampling_result.result,
        "content_preview": content[:160]
    }

@mcp.tool
async def interactive_project_brief(task: str, ctx: Context) -> dict:
    """Elicitation으로 추가 입력을 받고, Sampling으로 프로젝트 계획 초안을 만듭니다."""
    elicited = await ctx.elicit(
        message="프로젝트 계획 수립에 필요한 추가 정보를 입력해 주세요.",
        response_type=PlanInput
    )

    if elicited.action == "decline":
        return {
            "ok": False,
            "status": "declined",
            "message": "User declined to provide additional information."
        }

    if elicited.action == "cancel":
        return {
            "ok": False,
            "status": "cancelled",
            "message": "User cancelled the elicitation."
        }

    user_data = elicited.data

    prompt = f"""
Task: {task}
Deadline: {user_data.deadline}
Priority: {user_data.priority}
Assumptions: {user_data.assumptions}

Draft a short project brief in Korean with:
1) objective
2) constraints
3) first 3 next actions
""".strip()

    sampling_result = await ctx.sample(
        messages=prompt,
        system_prompt="You are a practical project planning assistant."
    )

    return {
        "ok": True,
        "task": task,
        "deadline": user_data.deadline,
        "priority": user_data.priority,
        "assumptions": user_data.assumptions,
        "brief": sampling_result.text or sampling_result.result
    }

print("MCP server 정의 완료")


## 3) Host 역할을 하는 Handler 정의

FastMCP Client는 callback handler를 통해 서버의 고급 요청에 응답할 수 있습니다.

- `sampling_handler`: 서버의 Sampling 요청을 처리
- `elicitation_handler`: 서버의 Elicitation 요청을 처리
- `roots_callback`: 서버가 요청할 때 roots 목록 제공

이번 실습에서는 실제 외부 LLM API 대신, **결정적(deterministic) 시뮬레이션 응답**을 사용합니다.


In [ ]:
async def demo_sampling_handler(messages, params, context):
    """
    서버가 요청한 Sampling을 처리하는 Host LLM 시뮬레이터.
    실제 API 호출 대신, 메시지를 읽어 간단한 요약/계획 문장을 생성합니다.
    """
    conversation = []
    for msg in messages:
        content = msg.content.text if hasattr(msg.content, "text") else str(msg.content)
        conversation.append((msg.role, content))

    user_texts = [text for role, text in conversation if role == "user"]
    latest = user_texts[-1] if user_texts else ""

    if "summarize" in latest.lower() or "요약" in latest:
        lines = [line.strip() for line in latest.splitlines() if line.strip()]
        body_lines = [ln for ln in lines if not ln.lower().startswith("please summarize")]
        preview = " / ".join(body_lines[:3])[:180]
        return f"요약 결과: {preview}"

    if "project brief" in latest.lower() or "next actions" in latest.lower():
        deadline = "unknown"
        priority = "unknown"
        for line in latest.splitlines():
            if line.lower().startswith("deadline:"):
                deadline = line.split(":", 1)[1].strip()
            if line.lower().startswith("priority:"):
                priority = line.split(":", 1)[1].strip()

        return (
            f"[프로젝트 브리프]\\n"
            f"- 목표: 요청된 과업을 {deadline} 일정 기준으로 준비합니다.\\n"
            f"- 우선순위: {priority}\\n"
            f"- 제약: 제공된 가정과 범위를 우선 반영합니다.\\n"
            f"- 다음 행동 1: 요구사항 정리\\n"
            f"- 다음 행동 2: 작업 분해 및 일정 배치\\n"
            f"- 다음 행동 3: 검토 포인트 정의"
        )

    return f"LLM 시뮬레이션 응답: {latest[:200]}"

async def demo_elicitation_handler(message, response_type, params, context):
    """
    서버의 Elicitation 요청을 처리하는 사용자 입력 시뮬레이터.
    이번 실습에서는 항상 accept 경로로 응답합니다.
    """
    print("=== ELICITATION REQUEST ===")
    print("message:", message)
    print("response_type:", response_type)

    if response_type is None:
        return {}

    if is_dataclass(response_type):
        values = {}
        for f in fields(response_type):
            if f.name == "deadline":
                values[f.name] = "2026-05-15"
            elif f.name == "priority":
                values[f.name] = "high"
            elif f.name == "assumptions":
                values[f.name] = "Demo workspace only; no external systems."
            else:
                values[f.name] = f"demo_{f.name}"
        return response_type(**values)

    # primitive type fallback
    if response_type is str:
        return "demo_value"
    if response_type is int:
        return 1
    if response_type is float:
        return 1.0
    if response_type is bool:
        return True

    return {}

async def roots_callback(context):
    """
    서버가 roots를 요청할 때 허용된 workspace 디렉터리를 반환합니다.
    """
    print("=== ROOTS REQUESTED BY SERVER ===")
    return [str(WORKSPACE_DIR)]

print("Handlers 정의 완료")


## 4) Client 생성

여기서 FastMCP Client가 **Host 역할**을 시뮬레이션합니다.

- `sampling_handler` 등록 → 서버의 LLM 요청 처리
- `elicitation_handler` 등록 → 서버의 사용자 입력 요청 처리
- `roots` 등록 → 서버에 허용된 파일 시스템 경계 제공


In [ ]:
client = Client(
    mcp,
    sampling_handler=demo_sampling_handler,
    elicitation_handler=demo_elicitation_handler,
    roots=roots_callback
)

print("Client 준비 완료")


## 5) Roots 확인

먼저 서버가 client roots를 실제로 볼 수 있는지 확인합니다.


In [ ]:
async def run_show_roots():
    async with client:
        result = await client.call_tool("show_roots", {})
        print("raw result:", result)
        print("data:", getattr(result, "data", None))
        print("content:", getattr(result, "content", None))

await run_show_roots()


## 6) 루트 내부 파일 읽기 + Sampling 요약

이번에는 roots 안에 있는 `project_note.txt`를 읽고,  
서버가 그 내용을 다시 Host의 LLM(sampling_handler)에 보내 요약하도록 합니다.


In [ ]:
async def run_sampling_demo():
    async with client:
        result = await client.call_tool(
            "summarize_note_from_root",
            {"filename": "project_note.txt"}
        )
        print("raw result:", result)
        print("data:", getattr(result, "data", None))
        print("content:", getattr(result, "content", None))

await run_sampling_demo()


## 7) Elicitation으로 추가 입력 수집 + Sampling으로 계획 생성

이 tool은 다음 순서로 동작합니다.

1. 서버가 사용자에게 추가 입력을 요청 (`ctx.elicit`)
2. 클라이언트의 `elicitation_handler`가 응답
3. 서버가 응답을 받아 prompt를 구성
4. 서버가 다시 `ctx.sample`로 Host LLM에 계획 초안 생성을 요청


In [ ]:
async def run_elicitation_demo():
    async with client:
        result = await client.call_tool(
            "interactive_project_brief",
            {"task": "Module 6 실습 교안 정리"}
        )
        print("raw result:", result)
        print("data:", getattr(result, "data", None))
        print("content:", getattr(result, "content", None))

await run_elicitation_demo()


## 8) 루트 경계 위반 시도 확인

이번에는 의도적으로 허용된 root 밖의 경로를 요청해 봅니다.  
서버가 root boundary를 기준으로 차단하는 예시입니다.


In [ ]:
async def run_roots_boundary_violation_demo():
    async with client:
        result = await client.call_tool(
            "summarize_note_from_root",
            {"filename": "../outside.txt"}
        )
        print("raw result:", result)
        print("data:", getattr(result, "data", None))
        print("content:", getattr(result, "content", None))

await run_roots_boundary_violation_demo()


## 9) 실습 정리

이번 실습에서 확인한 핵심:

1. **Sampling**은 서버가 클라이언트의 LLM 기능을 역으로 호출하는 패턴이다.
2. **Elicitation**은 서버가 실행 중간에 사용자에게 구조화된 추가 입력을 요청하는 패턴이다.
3. **Roots**는 클라이언트가 서버에 허용된 파일 시스템 경계를 알려 주는 기능이다.
4. 실제 Host가 없는 Colab에서도, FastMCP Client의 callback handler로 이 흐름을 시뮬레이션할 수 있다.
5. 세 기능은 서로 결합될 수 있다.  
   예: `Roots로 파일 범위 제한 → 파일 읽기 → Sampling 요약`,  
   또는 `Elicitation으로 파라미터 수집 → Sampling으로 계획 생성`
